#### 4. Evolve the table's schema two ways: append a new column using mergeSchema, then change an existing column's type using overwriteSchema; document the difference in what each requires.

In [0]:
%sql
select * from cyntexa_dev.bronze.simple_table

In [0]:
df = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/products_11_to_20.csv', inferSchema=True, header=True)
df.write.mode('append').options(mergeSchema=True).saveAsTable('cyntexa_dev.bronze.simple_table')

In [0]:
%sql
select * from cyntexa_dev.bronze.simple_table

In [0]:
df1 = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/products_21_to_30_price_as_string.csv', inferSchema=True, header=True)

df1.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cyntexa_dev.bronze.simple_table')

In [0]:
%sql
select * from cyntexa_dev.bronze.simple_table

#### 5. Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and confirm they're picked up automatically.

In [0]:
%sql
create or refresh streaming table cyntexa_dev.bronze.streaming_table
as select *, _metadata.file_path, _metadata.file_modification_time, current_timestamp() as ingest_ts from stream read_files('/Volumes/cyntexa_dev/sales/raw/streaming_dir/', format =>'csv', header => true);

```
resources:
  pipelines:
    pipeline_st_cyntexa_dev_bronze_streaming_table:
      name: ST-cyntexa_dev.bronze.streaming_table
      configuration:
        spark.sql.thriftServer.queryTimeout: "0"
        spark.databricks.sqlgateway.thriftServer.queryTimeout.maxTimeout: "172800"
        spark.databricks.photon.enabled: "true"
        spark.databricks.sqlgateway.thriftServer.queryTimeout.enabled: "true"
        spark.thriftserver.arrowBasedRowSet.timestampAsString: "true"
        spark.sql.session.timeZone: Etc/UTC
        spark.sql.ansi.enforceReservedKeywords: "false"
        spark.sql.ansi.enabled: "true"
        spark.sql.session.collation.default: UTF8_BINARY
        spark.sql.legacy.timeParserPolicy: CORRECTED
        spark.sql.ansi.enforceAnsiTypeCoercion: "true"
        pipelines.enzyme.mode: Advanced
        pipelines.decomposition.enabled: "true"
        pipelines.enzyme.enabled: "true"
        spark.databricks.streaming.dynamicAdmissionControl.cloudFiles.enabled: "true"
        pipelines.isDefaultStorageAndForcePreviewChannel: "true"
      managed_definition: {}
      target: bronze
      continuous: false
      development: true
      photon: true
      edition: ADVANCED
      catalog: cyntexa_dev
      serverless: true
```

#### 6. Use RESTORE to roll a table back to a version before a bad schema change, and describe what happens to the versions that were created after the point you restored to.

In [0]:
%sql
describe history cyntexa_dev.bronze.simple_table

In [0]:
%sql
restore table cyntexa_dev.bronze.simple_table to version as of 9

In [0]:
%sql
select * from cyntexa_dev.bronze.simple_table